In [1]:
from google import colab
colab.drive.mount('/content/drive')

Mounted at /content/drive


In [80]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [85]:
file_path = '/content/drive/MyDrive/portfolio projects/twitter_training.csv'
df = pd.read_csv(file_path, header=None)
df.rename(columns={3: 'tweets'}, inplace=True)
df = df[['tweets']]
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

In [146]:
df.head(5)

,tweets,cleaned_text
0,im getting on borderlands and i will murder yo...,im getting on borderlands and i will murder yo...
1,I am coming to the borders and I will kill you...,i am coming to the borders and i will kill you...
2,im getting on borderlands and i will kill you ...,im getting on borderlands and i will kill you ...
3,im coming on borderlands and i will murder you...,im coming on borderlands and i will murder you...
4,im getting on borderlands 2 and i will murder ...,im getting on borderlands 2 and i will murder ...


In [86]:
df = df[['tweets']][ :30000]

In [147]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30000 entries, 0 to 32257
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   tweets        30000 non-null  object
 1   cleaned_text  30000 non-null  object
dtypes: object(2)
memory usage: 703.1+ KB


In [87]:
def preprocess_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # Remove URLs
    text = re.sub(r'@\w+|#\w+', '', text)  # Remove mentions and hashtags
    text = re.sub(r'[^A-Za-z0-9\s\.\,\?\$\!\%]+', '', text)  # Remove special characters
    text = text.lower().strip()  # Convert to lowercase and strip leading/trailing spaces
    return text

In [88]:
df['cleaned_text'] = df['tweets'].apply(preprocess_text)

In [89]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['cleaned_text'])
total_vocab_size = len(tokenizer.word_index) + 1

In [90]:
def generate_sequences(text_reviews, tokenizer):
    input_seq = []
    for each_review in text_reviews:
        token_list = tokenizer.texts_to_sequences([each_review])[0]
        for i in range(0, len(token_list)-3):
            n_gram_seq = token_list[0:i+4]
            input_seq.append(n_gram_seq)
    return input_seq

In [91]:
input_sequences = generate_sequences(df['cleaned_text'], tokenizer)
max_length = max([len(seq) for seq in input_sequences])
input_sequences = pad_sequences(input_sequences, maxlen=max_length, padding="pre")

In [148]:
input_sequences[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,  28, 165,  14,  60], dtype=int32)

In [92]:
X_data = input_sequences[:, :-1]
y_data = input_sequences[:, -1]

In [149]:
X_data[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,  28, 165,  14], dtype=int32)

In [150]:
y_data[0]

np.int32(60)

In [93]:
X_data.shape

(462835, 165)

In [116]:
X_train, X_valid, y_train, y_valid = train_test_split(X_data, y_data, train_size=0.38, random_state=50, shuffle=True)

In [118]:
X_train.shape

(175877, 165)

In [119]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, train_size=0.80, random_state=50, shuffle=True)

In [120]:
X_train.shape, X_valid.shape

((140701, 165), (35176, 165))

In [121]:
def create_model(total_vocab_size, max_length):
    model = Sequential()
    model.add(Embedding(total_vocab_size, 300, input_length=max_length-1))
    model.add(LSTM(100))
    model.add(Dense(total_vocab_size, activation="softmax"))
    model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
    return model

In [126]:
# Create and train the model
model = create_model(total_vocab_size, max_length)

In [125]:
history = model.fit(X_train, y_train, epochs=2, validation_data=(X_valid, y_valid))

Epoch 1/2
4397/4397 ━━━━━━━━━━━━━━━━━━━━ 1755s 399ms/step - accuracy: 0.0561 - loss: 6.9400 - val_accuracy: 0.0993 - val_loss: 6.5176
Epoch 2/2
4397/4397 ━━━━━━━━━━━━━━━━━━━━ 1764s 401ms/step - accuracy: 0.1122 - loss: 6.0342 - val_accuracy: 0.1293 - val_loss: 6.2227


In [127]:
def plot_training_history(history):
    # Plot training & validation accuracy values
    plt.figure(figsize=(12, 6))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()

In [145]:
plot_training_history(history)

In [129]:
def generate_next_words(input_text, tokenizer, model, max_length, num_words=5):
    input_text = preprocess_text(input_text)
    token_list = tokenizer.texts_to_sequences([input_text])[0]
    for _ in range(num_words):
        input_seq = pad_sequences([token_list], maxlen=max_length-1, padding='pre')
        predicted = model.predict(input_seq, verbose=0)
        predicted_word_index = np.argmax(predicted)
        output_word = tokenizer.index_word[predicted_word_index]
        token_list.append(predicted_word_index)
    return ' '.join([tokenizer.index_word[i] for i in token_list])

In [144]:
input_text = "hi, how are you ?"  # Example input text
num_words = 2
next_words = generate_next_words(input_text, tokenizer, model, max_length, num_words)
print(f"{next_words}")

hi how are you have a
